# 워크북 2 — 선택형 PDF 챗봇 (강의자 완성본)
## 자료 선택
A: DocZoom Creator Pro 사용 안내 PDF
B: 2026학년도 동서울대학교 안전관리 계획서 PDF (85쪽)
C: 장애아동 관련 실태조사 및 종합적 지원체계 구축방안 연구 PDF
참고자료 https://app.notion.com/p/400b2e6bf807804e8a0ef17344369725
원본 PDF를 먼저 다운로드한 뒤 실습 1에서 업로드하세요. 자료 1과 4는 중복 링크입니다.
노션 학습자료의 실습 셀에 따라 TODO를 작성하고 순서대로 실행합니다.


## 실습 0. 환경 설정
아래 코드를 노션에서 복사하여 Colab의 [코드] 셀에 입력합니다. 실행 후 설치 오류가 없는지 확인합니다.
예상 결과: 설치 완료 (경고는 환경에 따라 다를 수 있음)



In [ ]:
%pip -q install pypdf scikit-learn gradio openai


## 실습 1. 제공 PDF 업로드·텍스트 추출
Drive 7주차 ZIP 안의 data/week07_course_guide.pdf를 Colab 파일에 업로드합니다. TODO에는 PDF 파일 이름을 입력합니다.
예상 결과: 텍스트가 있는 페이지: 2 (제공 PDF 기준)



## 자료를 직접 선택하세요
링크에서 PDF 파일 1개를 내려받습니다. 다음 코드 셀의 파일 업로드 창에 해당 PDF 1개만 선택하세요.
- [A 안내서](http://www.doczoom.co.kr/dist/manual/DocZoomCreatorPro_QuickStartGuide_ko.pdf)
- [B 안전관리계획](https://www.du.ac.kr/resources/AD/main/file/2026%ED%95%99%EB%85%84%EB%8F%84%20%EC%95%88%EC%A0%84%EA%B4%80%EB%A6%AC%20%EA%B3%84%ED%9A%8D%EC%84%9C.pdf)
- [C 연구보고서](https://repo.kicce.re.kr/bitstream/2019.oak/5161/2/CR2102.pdf)
페이지 수가 많으면 청크 수와 처리 시간이 늘어납니다.


In [ ]:
from pathlib import Path
from pypdf import PdfReader

# 실습 1: 제공된 자료 A/B/C 중 하나를 직접 내려받아 Colab에 업로드합니다.
# 반드시 본인이 선택한 PDF 한 개만 현재 작업 파일로 지정합니다.
from google.colab import files
uploaded = files.upload()
pdf_files = [name for name in uploaded if name.lower().endswith(".pdf")]
assert len(pdf_files) == 1, "PDF 한 개만 업로드하세요."
pdf_name = pdf_files[0]
pdf_path = Path(pdf_name)
assert pdf_path.exists()
print("선택한 PDF:", pdf_name)

reader = PdfReader(str(pdf_path))
records = []
for page_number, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").strip()
    if text:
        records.append({"page": page_number, "text": text})

assert records, "추출 텍스트가 없습니다. 스캔 PDF라면 OCR이 필요합니다."
print("텍스트가 있는 페이지:", len(records))\nprint("선택 문서:", pdf_name)


## 실습 2. 청크 생성
문서를 일정 단어 수로 나눕니다. TODO 2에서 size=75, overlap=20을 완성합니다.
예상 결과: 생성 청크: PDF 내용·추출 공백에 따른 정수



In [ ]:
def chunk_words(text, size=75, overlap=20):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("0 <= overlap < size 조건을 확인하세요.")
    words = text.split()
    step = size - overlap
    return [
        " ".join(words[i:i + size])
        for i in range(0, len(words), step)
        if words[i:i + size]
    ]

chunk_size = 75
overlap = 20
chunks = [
    {"page": item["page"], "text": part}
    for item in records
    for part in chunk_words(item["text"], chunk_size, overlap)
]
print("생성 청크:", len(chunks))
print("첫 청크:", chunks[0]["text"][:100])


## 실습 3. TF-IDF 문서 벡터 생성
한글에도 적용하기 쉬운 문자 단위 2~4그램 TF-IDF를 사용합니다. TODO에서 analyzer와 ngram_range를 완성합니다.
예상 결과: (청크 수, TF-IDF 특징 수) 형태



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4)
)
document_vectors = vectorizer.fit_transform(
    [part["text"] for part in chunks]
)
print("행렬 크기:", document_vectors.shape)


## 실습 4. 질문 검색 함수
질문을 같은 벡터 공간에 변환하고 코사인 유사도 순으로 상위 청크를 반환합니다. TODO에서 similarity 함수를 채웁니다.
예상 결과: 상위 두 개의 페이지, 소수점 3자리 점수, 근거 텍스트



In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(question, top_k=3):
    question_vector = vectorizer.transform([question])
    scores = cosine_similarity(
        question_vector, document_vectors
    ).ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return [
        {
            "page": chunks[int(i)]["page"],
            "text": chunks[int(i)]["text"],
            "score": float(scores[int(i)])
        }
        for i in order
    ]

results = retrieve("RAG가 무엇인가요?", top_k=2)
for item in results:
    print(f"페이지 {item['page']} / 점수 {item['score']:.3f}")
    print(item["text"][:100])


## 실습 5. 결과표·청크 크기 비교
3개 질문의 최고 검색 결과를 표로 만들고, size=45로 바꿔 실습 2~5를 재실행합니다.
예상 결과: 3행의 질문·페이지·유사도 테이블



In [ ]:
import pandas as pd

questions = [
    "RAG가 무엇인가요?",
    "API 키는 어떻게 관리하나요?",
    "청크를 왜 만드나요?"
]
rows = []
for question in questions:
    hit = retrieve(question, top_k=1)[0]
    rows.append({
        "질문": question,
        "페이지": hit["page"],
        "유사도": round(hit["score"], 3)
    })
result_table = pd.DataFrame(rows)
display(result_table)


## 실습 6. Gradio·API 선택 준비
워크북 2는 실습 0~5 다음에 수행합니다. 키가 없는 경우 검색만 수행합니다. 체크박스 기본값은 False입니다.
예상 결과: Gradio 준비 완료



In [ ]:
import os
import gradio as gr

try:
    from google.colab import userdata
    secret = userdata.get("OPENAI_API_KEY")
    if secret:
        os.environ["OPENAI_API_KEY"] = secret
except Exception:
    print("API 키 없이 검색 모드로 진행합니다.")

print("Gradio 준비 완료")


## 실습 7. 검색 근거로 답변 함수 구성
TODO에서 top_k=3을 입력합니다. 무료 모드에서는 근거가 출력되고, 키를 가진 학생만 생성 모드를 사용합니다.
예상 결과: 무료 검색 모드: 검색 근거를 확인하세요. (관련 근거가 있는 경우)



In [ ]:
def rag_answer(question, use_llm=False):
    if not (question or "").strip():
        return "질문을 입력하세요.", ""
    hits = retrieve(question, top_k=3)
    evidence = "\n\n".join(
        f"[PDF p.{hit['page']}, score {hit['score']:.3f}]\n{hit['text']}"
        for hit in hits
    )
    if hits[0]["score"] < 0.03:
        return "PDF에서 관련 근거를 확인할 수 없습니다.", evidence
    if not use_llm:
        return "무료 검색 모드: 검색 근거를 확인하세요.", evidence
    key = os.getenv("OPENAI_API_KEY")
    if not key:
        return "API 키가 없어 검색 모드로 실행했습니다.", evidence
    from openai import OpenAI
    try:
        response = OpenAI(api_key=key).responses.create(
            model="gpt-4.1-mini",
            instructions=(
                "제공된 문서 근거만 이용해 한국어로 답하고 "
                "페이지 번호를 표시하세요. 근거가 없다면 모른다고 하세요."
            ),
            input=f"질문: {question}\n\n근거:\n{evidence}"
        )
        return response.output_text, evidence
    except Exception as exc:
        return f"API 호출 실패: {type(exc).__name__}", evidence

print(rag_answer("RAG가 무엇인가요?")[0])


## 실습 8. Gradio UI 만들기
TODO에서 gr.Interface와 launch를 완성합니다. 브라우저에 표시된 URL 또는 Colab 내 UI에서 질문을 입력합니다.
예상 결과: PDF 질문창, LLM 사용 체크박스, 답변과 근거 영역이 표시됨



In [ ]:
app = gr.Interface(
    fn=rag_answer,
    inputs=[
        gr.Textbox(label="PDF 질문"),
        gr.Checkbox(label="LLM 생성 사용(선택)", value=False)
    ],
    outputs=[
        gr.Textbox(label="답변"),
        gr.Textbox(label="검색 근거", lines=12)
    ],
    title="7주차 PDF RAG 챗봇"
)
app.launch(share=False)


## 결과 제출
모든 셀을 실행한 후 결과표와 실행화면을 캡처하고 TODO를 완성한 노트북을 제출하세요.
